# Options from Zero · Episode 21: Rates options: Black-76 and Bachelier

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The market.** Series 2's illustrative market, valued on 25 September 2026: AONIA and BBSW curves (`quotes_illustrative.csv`), the SOFR curve (`quotes_sofr_illustrative.csv`), and AUD/USD spot, FX swap points and cross-currency basis (`quotes_fx_illustrative.csv`). Shaped like a plausible market, made up for teaching; not market data. Rates with Bip is not affiliated with the New York Fed.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The caplet · 4. The forward · 5. Black-76 · 6. Bachelier: volatility in basis points · 7. Different models, different wings · 8. Negative rates · 9. Check against QuantLib · 10. Part 2 checkpoint · 11. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_sofr_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_sofr_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_sofr_illustrative.csv').write_text("""instrument,label,value,unit
ois,1W,3.88,pct
ois,1M,3.87,pct
ois,2M,3.86,pct
ois,3M,3.85,pct
future,SR3 Dec 2026,96.20,price
future,SR3 Mar 2027,96.30,price
future,SR3 Jun 2027,96.39,price
future,SR3 Sep 2027,96.44,price
ois,2Y,3.66,pct
ois,3Y,3.63,pct
ois,5Y,3.63,pct
ois,7Y,3.70,pct
ois,10Y,3.81,pct
ois,15Y,3.94,pct
ois,20Y,3.98,pct
ois,30Y,3.96,pct
""")
print('wrote quotes_sofr_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_fx_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_fx_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_fx_illustrative.csv').write_text("""instrument,label,value,unit
fx,spot,0.71,rate
fx,ON,-0.098,pips
fx,TN,-0.033,pips
fx,1W,-0.23,pips
fx,1M,-1.1,pips
fx,2M,-2.66,pips
fx,3M,-4.29,pips
fx,6M,-10.68,pips
fx,9M,-19.65,pips
fx,1Y,-30.32,pips
xccy,2Y,14.75,bp
xccy,3Y,14.75,bp
xccy,4Y,15.25,bp
xccy,5Y,15.5,bp
xccy,7Y,15.5,bp
xccy,10Y,15.5,bp
""")
print('wrote quotes_fx_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
valuation_date = "2026-09-25"
aud_quotes_file = "quotes_illustrative.csv"
sofr_quotes_file = "quotes_sofr_illustrative.csv"
fx_quotes_file = "quotes_fx_illustrative.csv"
cap_strike = 0.0400   # 4.00%
black_vol = 0.18     # illustrative lognormal volatility
cap_notional = 10_000_000

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 21,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

In [ ]:
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": r * 100, "vol_pct": vol * 100}

Part 2's additions: Greeks, simulation, hedging, implied volatility, Black and Bachelier (`episodes/_shared/options_part2.py`):

In [ ]:
# Part 2 additions to options.py: the Greeks, simulation, a hedging simulator, implied volatility, and Black's and
# Bachelier's formulas on a forward (for FX and interest-rate options). Written out by hand and copied verbatim into
# each Part 2 notebook; the notebooks check them against QuantLib. Needs options.py (norm_cdf, black_scholes),
# math and numpy as np.

def norm_pdf(x):
    """The bell curve's height at x (standard normal density)."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def norm_cdf_v(x):
    """norm_cdf for a whole numpy array at once."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x, dtype=float) / math.sqrt(2)))

def bs_greeks(s, k, t, r, vol):
    """Price and Greeks of a European call on a share with no dividends.
    delta per $1 of share price, gamma per $1 squared, vega per 1.00 of volatility (100 vol points),
    theta per year (negative: value lost as time passes), rho per 1.00 of interest rate."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    disc = math.exp(-r * t)
    return {"price": s * norm_cdf(d1) - k * disc * norm_cdf(d2),
            "delta": norm_cdf(d1),
            "gamma": norm_pdf(d1) / (s * vol * math.sqrt(t)),
            "vega": s * norm_pdf(d1) * math.sqrt(t),
            "theta": -s * norm_pdf(d1) * vol / (2 * math.sqrt(t)) - r * k * disc * norm_cdf(d2),
            "rho": k * t * disc * norm_cdf(d2)}

def call_delta(s, k, t, r, vol):
    """Black-Scholes delta N(d1) for an array of share prices."""
    d1 = (np.log(np.asarray(s, dtype=float) / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    return norm_cdf_v(d1)

def simulate_paths(s, t, drift, vol, steps, n_paths, rng):
    """Share price paths: shape (n_paths, steps + 1), starting at s, lognormal steps with the given drift."""
    dt = t / steps
    z = rng.standard_normal((n_paths, steps))
    log_steps = (drift - vol ** 2 / 2) * dt + vol * math.sqrt(dt) * z
    return s * np.exp(np.concatenate([np.zeros((n_paths, 1)), np.cumsum(log_steps, axis=1)], axis=1))

def monte_carlo_call(s, k, t, r, vol, n, rng):
    """Price = the average discounted payoff over n risk-neutral outcomes, with its standard error."""
    z = rng.standard_normal(n)
    s_t = s * np.exp((r - vol ** 2 / 2) * t + vol * math.sqrt(t) * z)
    pv = math.exp(-r * t) * np.maximum(s_t - k, 0.0)
    return float(pv.mean()), float(pv.std(ddof=1) / math.sqrt(n))

def hedge_short_call(s, k, t, r, vol_implied, vol_realised, steps, n_paths, rng, hedge=True):
    """Sell the call at vol_implied and, if hedge, rebalance to delta shares (computed at vol_implied) `steps` times.
    The share moves with vol_realised and grows at r. Cash earns r. Returns each path's P&L at expiry in today's dollars."""
    dt = t / steps
    price = s * np.ones(n_paths)
    held = call_delta(price, k, t, r, vol_implied) if hedge else np.zeros(n_paths)
    cash = black_scholes(s, k, t, r, vol_implied) - held * price      # premium received, shares bought
    for i in range(1, steps + 1):
        price = price * np.exp((r - vol_realised ** 2 / 2) * dt + vol_realised * math.sqrt(dt) * rng.standard_normal(n_paths))
        cash = cash * math.exp(r * dt)
        if hedge and i < steps:
            new = call_delta(price, k, t - i * dt, r, vol_implied)
            cash = cash - (new - held) * price                        # buy (or sell) the difference
            held = new
    pnl = cash + held * price - np.maximum(price - k, 0.0)           # sell the shares, pay the option's payoff
    return pnl * math.exp(-r * t)

def implied_vol(price, s, k, t, r, lo=0.0001, hi=3.0, tol=1e-12):
    """Run Black-Scholes backwards by bisection: halve [lo, hi] until the price fits. Returns (vol, halvings)."""
    n = 0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if black_scholes(s, k, t, r, mid) < price:
            lo = mid
        else:
            hi = mid
        n += 1
    return (lo + hi) / 2, n

def black76(forward, strike, t, vol, df, call=True):
    """Black's formula on a forward: df x [F N(d1) - K N(d2)] for a call. FX options (Garman-Kohlhagen) and caplets."""
    sd = vol * math.sqrt(t)
    d1 = (math.log(forward / strike) + sd ** 2 / 2) / sd
    d2 = d1 - sd
    if call:
        return df * (forward * norm_cdf(d1) - strike * norm_cdf(d2))
    return df * (strike * norm_cdf(-d2) - forward * norm_cdf(-d1))

def bachelier(forward, strike, t, normal_vol, df, call=True):
    """Bachelier's formula: the forward moves by normal_vol x sqrt(t) in absolute terms (it can go below zero)."""
    sd = normal_vol * math.sqrt(t)
    d = (forward - strike) / sd
    if call:
        return df * ((forward - strike) * norm_cdf(d) + sd * norm_pdf(d))
    return df * ((strike - forward) * norm_cdf(-d) + sd * norm_pdf(d))

Series 2's market. The AUD curve code expects `today`, `cal` and `dc`:

In [ ]:
today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()
out["meta"]["valuation_date"] = valuation_date

AONIA and BBSW curves (Series 1):

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

The SOFR curve (Series 2, Episode 3):

In [ ]:
# SOFR curve used from Series 2 Episode 3 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today (the evaluation date) from the setup cell.
#   SOFR OIS: CFTC Swaps Made Available to Trade chart (USD SOFR OIS): spot T+2, annual fixed and floating
#     legs, ACT/360, payment lag 2 days, New York business days; SOFR fixes on the government securities calendar.
#   3M SOFR futures: CME Submission 18-069: SOFR compounded over the IMM-to-IMM Reference Quarter,
#     price = 100 - rate; named by the month the Reference Quarter starts. Convexity set to zero (a simplification).
USD_CAL = ql.UnitedStates(ql.UnitedStates.Settlement)  # stand-in for New York business days
USD_DC = ql.Actual360()
MONTHS = {m: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def sofr_ois_helper(tenor, quote):
    return ql.OISRateHelper(2, ql.Period(tenor), quote, ql.Sofr(), paymentLag=2,
                            paymentFrequency=ql.Annual, paymentCalendar=USD_CAL,
                            convention=ql.ModifiedFollowing, endOfMonth=True)

def build_sofr_curve(quotes):
    """quotes: rows (instrument, label, value); OIS values in %, futures as prices. Returns curve, quotes, helpers."""
    q, helpers = {}, {}
    for r in quotes.itertuples():
        if r.instrument == "future":  # label like "SR3 Dec 2026"
            q[r.label] = ql.SimpleQuote(r.value)
            _, mon, year = r.label.split()
            helpers[r.label] = ql.SofrFutureRateHelper(ql.QuoteHandle(q[r.label]), MONTHS[mon], int(year),
                                                       ql.Quarterly)
        else:
            q[r.label] = ql.SimpleQuote(r.value / 100)
            helpers[r.label] = sofr_ois_helper(r.label, ql.QuoteHandle(q[r.label]))
    curve = ql.PiecewiseLogLinearDiscount(today, list(helpers.values()), USD_DC)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now: the helpers only see the curve once it has been built
    return curve, q, helpers

AUD/USD spot, forwards and the AUD curve under USD collateral (Series 2, Episodes 5-8):

In [ ]:
# AUD discount curve under USD collateral, used from Series 2 Episode 6 on. Copied verbatim into each notebook by
# make_notebook.py, so every notebook runs on its own. Needs: ql, pd, today, cal, dc and the AUD curve code
# (aud_curves.py) and SOFR curve code (sofr_curve.py) from earlier cells.
#   Short end: AUD/USD FX swaps, overnight to one year (forward = spot + points).
#   Long end: BBSW/SOFR mark-to-market cross-currency basis swaps, two to ten years (AFMA term sheet, 2024):
#     spot T+2 on Sydney and New York days, quarterly; receive 3M BBSW + spread on a constant AUD notional,
#     pay SOFR compounded with no spread on a USD notional reset each quarter to the forward FX rate.
#   Collateral: US dollars, discounted at SOFR (AFMA). The curve solved for is the AUD one.
#   Modelling choice: QuantLib's helper pays coupons and notionals on the same dates. A payment lag of zero
#   puts the notional exchanges on the roll dates, as in the term sheet; the coupons then pay two days early,
#   which moves the fair spread by less than a tenth of a basis point here.
JOINT = ql.JointCalendar(ql.Australia(ql.Australia.Settlement), USD_CAL)
FX_TENORS = ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]

class Market:
    """All curves of the series: AONIA and BBSW (AUD series), SOFR (Episode 3), AUD under USD collateral."""

    def __init__(self, aud_quotes, sofr_quotes, fx_quotes):
        _, handles, self.aud_q, _ = build_curves(aud_quotes)
        self.aonia, self.bbsw3m = handles["AONIA"], handles["BBSW3M"]
        self.sofr_curve, self.sofr_q, _ = build_sofr_curve(sofr_quotes)
        self.sofr = ql.YieldTermStructureHandle(self.sofr_curve)
        self.tom, self.spot = JOINT.advance(today, 1, ql.Days), JOINT.advance(today, 2, ql.Days)
        fx = fx_quotes[fx_quotes.instrument == "fx"].set_index("label").value
        xc = fx_quotes[fx_quotes.instrument == "xccy"].set_index("label").value
        self.S = ql.SimpleQuote(float(fx["spot"]))
        self.points = {t: ql.SimpleQuote(float(fx[t]) / 1e4) for t in ["ON", "TN"] + FX_TENORS}
        self.basis = {t: ql.SimpleQuote(float(v) / 1e4) for t, v in xc.items()}
        self.near_on, self.near_tn = ql.SimpleQuote(0.0), ql.SimpleQuote(0.0)
        self._set_near()
        h = ql.QuoteHandle
        self.helpers = {
            "ON": ql.FxSwapRateHelper(h(self.points["ON"]), h(self.near_on), ql.Period(1, ql.Days), 0, JOINT,
                                      ql.Following, False, False, self.sofr),
            "TN": ql.FxSwapRateHelper(h(self.points["TN"]), h(self.near_tn), ql.Period(1, ql.Days), 1, JOINT,
                                      ql.Following, False, False, self.sofr)}
        for t in FX_TENORS:
            self.helpers[t] = ql.FxSwapRateHelper(h(self.points[t]), h(self.S), ql.Period(t), 2, JOINT,
                                                  ql.ModifiedFollowing, True, False, self.sofr)
        for t, q in self.basis.items():
            self.helpers["basis " + t] = ql.MtMCrossCurrencyBasisSwapRateHelper(
                h(q), ql.Period(t), 2, JOINT, ql.ModifiedFollowing, True,
                bbsw(3, self.bbsw3m), ql.Sofr(self.sofr), self.sofr,
                False,   # isFxBaseCurrencyCollateralCurrency: AUD is the base currency; collateral is USD
                True,    # isBasisOnFxBaseCurrencyLeg: the spread is on the AUD (BBSW) leg
                False,   # isFxBaseCurrencyLegResettable: the USD notional resets, the AUD one is constant
                ql.Quarterly, 0, ql.Quarterly)
        self.curve = ql.PiecewiseLogLinearDiscount(today, list(self.helpers.values()), dc)
        self.curve.enableExtrapolation()
        self.curve.nodes()
        self.aud_usd = ql.YieldTermStructureHandle(self.curve)

    def _set_near(self):  # today's and tomorrow's rates, built back from spot with the ON and TN points
        self.near_tn.setValue(self.S.value() - self.points["TN"].value())
        self.near_on.setValue(self.S.value() - self.points["TN"].value() - self.points["ON"].value())

    def set_spot(self, value):
        self.S.setValue(value)
        self._set_near()

    def fwd_fx(self, d):
        """AUD/USD outright for value date d: spot x P_aud(spot, d) / P_usd(spot, d)."""
        a, u = self.aud_usd, self.sofr
        return self.S.value() * (a.discount(d) / a.discount(self.spot)) / (u.discount(d) / u.discount(self.spot))

def mtm_swap(m, tenor, spread, aud_notional, coupon_lag=0):
    """Value a BBSW/SOFR mark-to-market basis swap to the party receiving the AUD leg (BBSW + spread).

    Dates as the curve's helpers build them: spot + tenor, rolled backwards quarterly, end-of-month rule.
    Each quarter the USD notional is the AUD notional times the forward FX rate to the period start; notionals
    move on roll dates; coupons are paid coupon_lag business days after each period end (term sheet: 2; the
    curve build: 0). Values are in AUD at today's FX rate. Returns the value, the value of one basis point of
    spread, and one row per period.
    """
    sch = list(ql.MakeSchedule(m.spot, m.spot + ql.Period(tenor), ql.Period(ql.Quarterly), calendar=JOINT,
                               convention=ql.ModifiedFollowing, endOfMonth=True, backwards=True))
    pa, pu = m.aud_usd.discount, m.sofr.discount
    fx_today = m.fwd_fx(today)
    index = bbsw(3, m.bbsw3m)
    pv_aud = -aud_notional * pa(sch[0]) + aud_notional * pa(sch[-1])
    pv_usd, bps, rows = 0.0, 0.0, []
    for i, (a, b) in enumerate(zip(sch[:-1], sch[1:])):
        pay = JOINT.advance(b, coupon_lag, ql.Days) if coupon_lag else b
        tau_a, tau_u = dc.yearFraction(a, b), USD_DC.yearFraction(a, b)
        bbsw_rate = (m.bbsw3m.discount(a) / m.bbsw3m.discount(b) - 1) / tau_a
        sofr_rate = (pu(a) / pu(b) - 1) / tau_u
        fx = m.fwd_fx(a)
        usd_n = aud_notional * fx
        aud_cpn = aud_notional * (bbsw_rate + spread) * tau_a
        usd_cpn = usd_n * sofr_rate * tau_u
        pv_aud += aud_cpn * pa(pay)
        bps += aud_notional * tau_a * pa(pay) * 1e-4
        pv_usd += usd_n * (pu(a) - pu(b)) - usd_cpn * pu(pay)   # receive the USD notional, repay it with SOFR
        rows.append({"period": i + 1, "start": a.ISO(), "end": b.ISO(), "pay": pay.ISO(),
                     "bbsw_pct": bbsw_rate * 100, "aud_rate_pct": (bbsw_rate + spread) * 100, "aud_coupon": aud_cpn,
                     "fx": fx, "usd_notional": usd_n, "sofr_pct": sofr_rate * 100, "usd_coupon": usd_cpn})
    value = pv_aud + pv_usd / fx_today
    return {"value": value, "bps": bps, "pv_aud_leg": pv_aud, "pv_usd_leg_aud": pv_usd / fx_today, "rows": rows}

def par_spread(m, tenor, coupon_lag=0):
    """Spread on the AUD leg that makes a new mark-to-market basis swap worth zero."""
    r = mtm_swap(m, tenor, 0.0, 1.0, coupon_lag)
    return -r["value"] / (r["bps"] * 1e4)

In [ ]:
m = Market(pd.read_csv(aud_quotes_file), pd.read_csv(sofr_quotes_file), pd.read_csv(fx_quotes_file))
print("today", today, "| spot date", m.spot, "| AUD/USD", m.S.value())

## 2. Last episode's question

If Australian interest rates rose, the AUD call would get **cheaper**: a higher AUD rate lowers the forward $F = S\,P_{AUD}/P_{USD}$. In Episode 20's notebook, a 0.50% rise in the AUD curve cut the call by about USD 339.

In [ ]:
out["answer"] = {"direction": "cheaper", "bump_pct": 0.5}
out["answer"]

## 3. The caplet

A **caplet** pays $\max(L - K, 0)\times\tau\times N$, where $L$ is 3-month BBSW set at the start of a future period, $\tau$ the period's year fraction (ACT/365F) and $N$ the notional. It is paid at the end of the period (set in advance, paid in arrears). A **cap** is a string of caplets: a borrower paying BBSW buys one to put a ceiling on their rate. Ours: AUD 10 million, strike 4.00%, on the 3-month period starting one year from today (Sydney calendar).

In [ ]:
start_ = cal.advance(today, ql.Period(1, ql.Years), ql.ModifiedFollowing)
end_ = cal.advance(start_, ql.Period(3, ql.Months), ql.ModifiedFollowing)
tau = dc.yearFraction(start_, end_)
T_fix = dc.yearFraction(today, start_)
out["caplet"] = {"today": today.ISO(), "start": start_.ISO(), "end": end_.ISO(), "tau": tau, "T": T_fix, "strike_pct": cap_strike * 100,
                 "notional": cap_notional, "vol_pct": black_vol * 100, "days": end_ - start_}
out["caplet"]

## 4. The forward

From Series 1's curves: the forward 3-month BBSW rate for that period comes from the BBSW 3M curve, and the payment is discounted on the AONIA curve (the dual-curve rule of Series 1, Episode 5).

In [ ]:
F = m.bbsw3m.forwardRate(start_, end_, dc, ql.Simple).rate()
df_pay = m.aonia.discount(end_)
out["forward"] = {"forward_pct": F * 100, "df": df_pay, "annuity": cap_notional * tau * df_pay}
out["forward"]

## 5. Black-76

Black (1976) priced options on futures and forwards by treating the forward as lognormal. For a caplet, the forward rate is the "price":

$$\text{caplet} = N\,\tau\,P(0,t_{pay})\,\big[F\,N(d_1) - K\,N(d_2)\big], \qquad d_{1,2} = \frac{\ln(F/K) \pm \tfrac12\sigma^2T}{\sigma\sqrt{T}}$$

with $T$ the time to the fixing. Volatility here is *lognormal*: a percentage of the rate.

In [ ]:
black = cap_notional * tau * black76(F, cap_strike, T_fix, black_vol, df_pay)
out["black"] = {"price": black, "price_bp_running": black / (cap_notional * tau * df_pay) * 1e4}
out["black"]

## 6. Bachelier: volatility in basis points

Bachelier (1900, Episode 11) let prices move by normal amounts. For rates, that means volatility in **basis points**: the forward rate moves up or down by $\sigma_N\sqrt T$ in absolute terms.

$$\text{caplet} = N\,\tau\,P\,\big[(F - K)\,N(d) + \sigma_N\sqrt T\,n(d)\big], \qquad d = \frac{F - K}{\sigma_N\sqrt T}$$

The normal volatility that gives the same price as Black at 18%, and the rule of thumb $\sigma_N \approx \sigma_B \times F$:

In [ ]:
def price_normal(sn, k=cap_strike):
    return cap_notional * tau * bachelier(F, k, T_fix, sn, df_pay)
lo, hi = 1e-6, 0.05
for _ in range(200):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if price_normal(mid) < black else (lo, mid)
normal_vol = (lo + hi) / 2
out["bachelier"] = {"normal_vol_bp": normal_vol * 1e4, "rule_of_thumb_bp": black_vol * F * 1e4, "price": price_normal(normal_vol)}
assert abs(out["bachelier"]["price"] - black) < 1e-6
out["bachelier"]

## 7. Different models, different wings

Same price at the 4.00% strike, but at other strikes the two models disagree: Black's lognormal forward has a long upper tail (high strikes dearer) and can't go below zero (low strikes cheaper than Bachelier's). Choosing a model is choosing a smile, which is where Part 3 begins.

In [ ]:
rows = []
for k in (0.030, 0.035, 0.040, 0.045, 0.050):
    b = cap_notional * tau * black76(F, k, T_fix, black_vol, df_pay)
    n_ = price_normal(normal_vol, k)
    rows.append({"strike_pct": k * 100, "black": b, "bachelier": n_, "diff": b - n_})
out["wings"] = {"rows": rows, "k5_black": rows[-1]["black"], "k5_bachelier": rows[-1]["bachelier"],
                "k3_black": rows[0]["black"], "k3_bachelier": rows[0]["bachelier"]}
pd.DataFrame(rows)

## 8. Negative rates

Policy rates have gone below zero: the ECB's deposit facility rate was negative from 11 June 2014 (−0.10%) to 27 July 2022, reaching −0.50%; the Bank of Japan applied −0.1% to part of banks' current accounts from its decision of 29 January 2016 until 19 March 2024. Black's formula needs $\ln(F/K)$, which doesn't exist for a negative forward. Bachelier's doesn't care. A one-year 0% cap on AUD 10 million of 3-month periods is priced here as a single caplet on a forward of −0.20% with 60bp normal volatility.

In [ ]:
neg = {"forward_pct": -0.20, "forward_abs_pct": 0.20, "strike_pct": 0.0, "normal_vol_bp": 60, "T": 1.0, "tau": 0.25, "notional": cap_notional}
neg["price"] = cap_notional * 0.25 * bachelier(-0.0020, 0.0, 1.0, 0.0060, 1.0)
try:
    black76(-0.0020, 0.0001, 1.0, 0.18, 1.0)
    neg["black_works"] = True
except ValueError as e:
    neg["black_works"] = False
    neg["black_error"] = str(e)
out["negative"] = neg
out["history"] = {"ecb_start": 2014, "ecb_end": 2022, "ecb_low_pct": -0.50, "boj_start": 2016, "boj_end": 2024, "boj_rate_pct": -0.1,
                  "black_year": 1976, "bachelier_year": 1900}
out["negative"]

## 9. Check against QuantLib

`blackFormula` and `bachelierBlackFormula` (undiscounted, per unit of forward), and a one-period `CapFloor` on QuantLib's BBSW 3M index with `BlackCapFloorEngine` and `BachelierCapFloorEngine`.

In [ ]:
std = black_vol * math.sqrt(T_fix)
ql_black = cap_notional * tau * df_pay * ql.blackFormula(ql.Option.Call, cap_strike, F, std)
ql_bach = cap_notional * tau * df_pay * ql.bachelierBlackFormula(ql.Option.Call, cap_strike, F, normal_vol * math.sqrt(T_fix))
index = ql.Bbsw3M(m.bbsw3m)
schedule = ql.Schedule(start_, end_, ql.Period(3, ql.Months), cal, ql.ModifiedFollowing, ql.ModifiedFollowing,
                       ql.DateGeneration.Forward, False)
leg = ql.IborLeg([cap_notional], schedule, index, dc, ql.ModifiedFollowing, fixingDays=[0])
cap = ql.Cap(leg, [cap_strike])
cap.setPricingEngine(ql.BlackCapFloorEngine(m.aonia, ql.QuoteHandle(ql.SimpleQuote(black_vol)), dc))
cap_black = cap.NPV()
cap.setPricingEngine(ql.BachelierCapFloorEngine(m.aonia, ql.QuoteHandle(ql.SimpleQuote(normal_vol))))   # ACT/365F by default
cap_bach = cap.NPV()
cpn = ql.as_floating_rate_coupon(leg[0])
out["quantlib"] = {"black_formula": ql_black, "bachelier_formula": ql_bach, "cap_black": cap_black, "cap_bachelier": cap_bach,
                   "coupon_fixing": cpn.fixingDate().ISO(), "coupon_forward_pct": cpn.indexFixing() * 100}
assert abs(ql_black - black) < 1e-6 and abs(ql_bach - black) < 1e-6
assert abs(cap_black - black) < 0.01 and abs(cap_bach - black) < 0.01
out["quantlib"]

## 10. Part 2 checkpoint (answers)

1. Explain today's P&L: $\Delta = 0.5$, $\Gamma = 0.02$, vega 0.4 per point, $\Theta = -0.015$ a day; the share rose \$2 and volatility fell 1 point.
2. You sold an option at 20% volatility, hedged daily, and realised came in at 18%. Profit or loss, and roughly how much with vega 40 (per 1.00)?
3. Why does the AUD/USD call use the AUD interest rate at all, if the premium is paid in USD?

In [ ]:
q1 = 0.5 * 2 + 0.5 * 0.02 * 2 ** 2 + 0.4 * (-1) + (-0.015)
q2 = 40 * (0.20 - 0.18)
out["checkpoint"] = {"q1": q1, "q2": q2,
                     "q3": "Holding AUD earns the AUD rate, like a dividend: it sets the forward F = S x P_AUD / P_USD, which the option prices off."}
out["checkpoint"]

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")